# QLoRA clean 30k from scratch

Цель: обучить Qwen QLoRA с нуля на 30k train examples, но без лишних долгих ячеек.

Оптимизации:
- без base/zero-shot evaluation;
- без train_eval;
- adapter сохраняется сразу после training;
- evaluation только val/test;
- val/test по умолчанию 1000 examples each, чтобы не умереть на долгой evaluation.


In [ ]:
!pip install -q -U "transformers>=4.45.0" accelerate peft bitsandbytes scikit-learn pandas pyarrow safetensors


In [ ]:
import os, json, glob, math, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("device count:", torch.cuda.device_count())


## 1. Конфиг

Это главный безопасный вариант:

```text
30k train = 15k Yes + 15k No
1 epoch
eval: 1000 val + 1000 test
```

Если после training останется много времени, можно отдельно поднять `EVAL_PER_CLASS = 1000` и пересчитать eval.


In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

MAX_SEQ_LENGTH = 1024
TRAIN_PER_CLASS = 15000       # 30k train total
EVAL_PER_CLASS = 500          # 1000 val/test total. Для точнее можно 1000, но это дольше.

NUM_EPOCHS = 1
LR = 2e-4
GRAD_ACCUM = 8
BATCH_SIZE = 1
SAVE_STEPS = 1000

SKIP_EVAL = False             # если сессия подходит к лимиту, можно True
RUN_NAME = f"qwen_clean_30k_from_scratch_eval{EVAL_PER_CLASS*2}"
OUT_DIR = Path("/kaggle/working") / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("run:", RUN_NAME)
print("out:", OUT_DIR)


## 2. Поиск instruction_temporal


In [ ]:
def find_file(filename, prefer_substr="instruction"):
    hits = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"Не найден {filename} в /kaggle/input")
    hits = sorted(hits, key=lambda p: (prefer_substr.lower() not in p.lower(), len(p)))
    return Path(hits[0])

train_path = find_file("train.jsonl")
val_path = find_file("val.jsonl")
test_path = find_file("test.jsonl")

print("train:", train_path)
print("val:", val_path)
print("test:", test_path)


## 3. Загрузка данных и balanced samples


In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_df = load_jsonl(train_path)
val_df = load_jsonl(val_path)
test_df = load_jsonl(test_path)

print("shapes:", train_df.shape, val_df.shape, test_df.shape)
print("train output:", train_df["output"].value_counts(dropna=False).to_dict())

def class_mask(df, cls):
    return df["output"].astype(str).str.strip().str.lower() == cls.lower()

def make_balanced_sample(df, n_per_class, seed=42):
    yes = df[class_mask(df, "yes")].sample(n=n_per_class, random_state=seed)
    no = df[class_mask(df, "no")].sample(n=n_per_class, random_state=seed)
    return pd.concat([yes, no], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)

train_pilot_df = make_balanced_sample(train_df, TRAIN_PER_CLASS, SEED)
val_pilot_df = make_balanced_sample(val_df, EVAL_PER_CLASS, SEED)
test_pilot_df = make_balanced_sample(test_df, EVAL_PER_CLASS, SEED)

print("train:", train_pilot_df.shape, train_pilot_df["output"].value_counts().to_dict())
print("val:", val_pilot_df.shape, val_pilot_df["output"].value_counts().to_dict())
print("test:", test_pilot_df.shape, test_pilot_df["output"].value_counts().to_dict())


## 4. Tokenizer, prompt, answer-only dataset


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

SYSTEM_PROMPT = "You are a recommendation model. Answer only Yes or No."

def normalize_answer(x):
    x = str(x).strip()
    if x.lower().startswith("yes"):
        return "Yes"
    if x.lower().startswith("no"):
        return "No"
    raise ValueError(f"Bad output: {x}")

def build_user_text(row):
    instruction = str(row.get("instruction", "")).strip()
    inp = str(row.get("input", "")).strip()
    if instruction and inp:
        return instruction + "\n\n" + inp
    if "prompt_text" in row and pd.notna(row["prompt_text"]):
        return str(row["prompt_text"]).strip()
    raise ValueError("Нет instruction/input и нет prompt_text")

def make_prompt_text(row):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_text(row)}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def make_answer_text(row):
    return normalize_answer(row["output"]) + tokenizer.eos_token

print(make_prompt_text(train_pilot_df.iloc[0])[-1200:])
print("answer:", make_answer_text(train_pilot_df.iloc[0]))


In [ ]:
class RecSFTDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_length=1024):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_ids = self.tokenizer(make_prompt_text(row), add_special_tokens=False).input_ids
        answer_ids = self.tokenizer(make_answer_text(row), add_special_tokens=False).input_ids

        max_prompt_len = self.max_length - len(answer_ids)
        if len(prompt_ids) > max_prompt_len:
            prompt_ids = prompt_ids[-max_prompt_len:]

        input_ids = prompt_ids + answer_ids
        labels = [-100] * len(prompt_ids) + answer_ids
        attention_mask = [1] * len(input_ids)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long)
        }

def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attention_mask, labels = [], [], []
    for x in batch:
        pad_len = max_len - len(x["input_ids"])
        input_ids.append(torch.cat([x["input_ids"], torch.full((pad_len,), tokenizer.pad_token_id, dtype=torch.long)]))
        attention_mask.append(torch.cat([x["attention_mask"], torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(torch.cat([x["labels"], torch.full((pad_len,), -100, dtype=torch.long)]))
    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attention_mask),
        "labels": torch.stack(labels)
    }

train_ds = RecSFTDataset(train_pilot_df, tokenizer, MAX_SEQ_LENGTH)

lens = [len(train_ds[i]["input_ids"]) for i in range(min(1000, len(train_ds)))]
print("length min/mean/max:", min(lens), float(np.mean(lens)), max(lens))
print("train examples:", len(train_ds))


## 5. Загрузка модели и LoRA


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 6. Training и немедленное сохранение adapter


In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUT_DIR / "trainer"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    fp16=True,
    logging_steps=20,
    save_steps=SAVE_STEPS,
    save_strategy="steps",
    save_total_limit=3,
    report_to="none",
    optim="paged_adamw_8bit",
    seed=SEED,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    warmup_ratio=0.03
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collate_fn
)

print("approx optimization steps:", math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS)
trainer.train()

adapter_dir = OUT_DIR / "adapter_qwen_clean_30k"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
shutil.make_archive(str(OUT_DIR / "adapter_qwen_clean_30k"), "zip", adapter_dir)

print("adapter saved:", OUT_DIR / "adapter_qwen_clean_30k.zip")


## 7. Evaluation только val/test


In [ ]:
@torch.no_grad()
def answer_logprob(model, prompt_text, answer_text):
    model.eval()
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
    answer_ids = tokenizer(answer_text, add_special_tokens=False).input_ids
    max_prompt_len = MAX_SEQ_LENGTH - len(answer_ids)
    if len(prompt_ids) > max_prompt_len:
        prompt_ids = prompt_ids[-max_prompt_len:]
    input_ids = torch.tensor([prompt_ids + answer_ids], dtype=torch.long, device=model.device)
    attention_mask = torch.ones_like(input_ids)
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = out.logits[0]
    start = len(prompt_ids)
    total = 0.0
    for j, tok_id in enumerate(answer_ids):
        pos = start + j - 1
        total += float(torch.log_softmax(logits[pos], dim=-1)[tok_id].detach().cpu())
    return total

@torch.no_grad()
def predict_scores(model, df):
    scores, y_true, sample_ids = [], [], []
    yes_answer = "Yes" + tokenizer.eos_token
    no_answer = "No" + tokenizer.eos_token
    for i, (_, row) in enumerate(df.iterrows(), start=1):
        prompt_text = make_prompt_text(row)
        lp_yes = answer_logprob(model, prompt_text, yes_answer)
        lp_no = answer_logprob(model, prompt_text, no_answer)
        m = max(lp_yes, lp_no)
        p_yes = math.exp(lp_yes - m) / (math.exp(lp_yes - m) + math.exp(lp_no - m))
        scores.append(p_yes)
        y_true.append(1 if normalize_answer(row["output"]) == "Yes" else 0)
        sample_ids.append(row.get("sample_id", None))
        if i % 250 == 0:
            print("processed", i, "/", len(df))
    return np.array(y_true), np.array(scores), sample_ids

def best_threshold_by_f1(y_true, scores):
    best = {"threshold": 0.5, "f1": -1}
    for thr in np.linspace(0.01, 0.99, 99):
        pred = (scores >= thr).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(y_true, pred, average="binary", zero_division=0)
        if f1 > best["f1"]:
            best = {"threshold": float(thr), "precision": float(p), "recall": float(r), "f1": float(f1)}
    return best

def calc_metrics(y_true, scores, threshold):
    pred = (scores >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, pred, average="binary", zero_division=0)
    return {
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "pr_auc": float(average_precision_score(y_true, scores)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f1),
        "threshold": float(threshold)
    }

def evaluate_split(model, df, name, threshold=None):
    y_true, scores, sample_ids = predict_scores(model, df)
    if threshold is None:
        threshold = best_threshold_by_f1(y_true, scores)["threshold"]
    metrics = calc_metrics(y_true, scores, threshold)
    print(name, metrics)
    pred_df = pd.DataFrame({"sample_id": sample_ids, "y_true": y_true, "score_yes": scores, "pred": (scores >= threshold).astype(int)})
    return metrics, pred_df


In [ ]:
if not SKIP_EVAL:
    val_metrics, val_pred = evaluate_split(model, val_pilot_df, "qlora30k_val")
    thr = val_metrics["threshold"]
    test_metrics, test_pred = evaluate_split(model, test_pilot_df, "qlora30k_test", threshold=thr)

    summary = pd.DataFrame([
        {"stage": "qwen_clean_30k_from_scratch", "split": "val", **val_metrics},
        {"stage": "qwen_clean_30k_from_scratch", "split": "test", **test_metrics},
    ])
    summary.to_csv(OUT_DIR / "qwen_clean_30k_metrics.csv", index=False)
    val_pred.to_csv(OUT_DIR / "qwen_clean_30k_val_predictions.csv", index=False)
    test_pred.to_csv(OUT_DIR / "qwen_clean_30k_test_predictions.csv", index=False)
    display(summary)
else:
    summary = pd.DataFrame()
    print("SKIP_EVAL=True: evaluation skipped")


## 8. Summary и архив


In [ ]:
run_summary = {
    "model_name": MODEL_NAME,
    "train_per_class": int(TRAIN_PER_CLASS),
    "train_examples": int(len(train_pilot_df)),
    "eval_per_class": int(EVAL_PER_CLASS),
    "val_examples": int(len(val_pilot_df)),
    "test_examples": int(len(test_pilot_df)),
    "num_epochs": int(NUM_EPOCHS),
    "lr": float(LR),
    "max_seq_length": int(MAX_SEQ_LENGTH),
    "skip_eval": bool(SKIP_EVAL),
    "metrics": summary.to_dict(orient="records") if len(summary) else []
}

with open(OUT_DIR / "qwen_clean_30k_summary.json", "w", encoding="utf-8") as f:
    json.dump(run_summary, f, ensure_ascii=False, indent=2)

shutil.make_archive(str(OUT_DIR / "qwen_clean_30k_full_outputs"), "zip", OUT_DIR)

print("saved:", OUT_DIR)
print("artifact:", OUT_DIR / "adapter_qwen_clean_30k.zip")
print("artifact:", OUT_DIR / "qwen_clean_30k_full_outputs.zip")


## Output artifacts

Main artifacts:

```text
adapter_qwen_clean_30k.zip
qwen_clean_30k_full_outputs.zip
qwen_clean_30k_summary.json
```

Evaluation artifacts, if evaluation is completed:

```text
qwen_clean_30k_metrics.csv
qwen_clean_30k_test_predictions.csv
qwen_clean_30k_val_predictions.csv
```

The adapter is saved immediately after training.
